# LMA_Assignment


## 1. Setup

In [1]:
import re, time, os
import numpy as np, pandas as pd, pypdf, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

BI, CE = "Qwen/Qwen3-Embedding-4B", "BAAI/bge-reranker-v2-m3"
LLM_URL = os.environ.get("LLM_URL", "http://10.20.204.252:8011/v1")  # Qwen3.8-27B served by vLLM
os.makedirs("results", exist_ok=True)

## 2. Chunk: one chunk per numbered rule item (e.g. `12-26`)

In [2]:
pages = pypdf.PdfReader("data/fiba_obri_2024.pdf").pages[4:]  # skip cover, intro, table of contents
text = " ".join(p.extract_text() for p in pages)
text = re.sub(r"October 2024, version 1\.0a 2024 OFFICIAL BASKETBALL RULES INTERPRETATIONS Page \d+ of 142", " ", text)
text = re.sub(r"\s+", " ", text)
ITEM_ID = r"(?<![\w/.-])((?:\d{1,2}(?:/\d{1,2})?|[A-F])-\d{1,3}(?:\.\d{1,3}){0,2}) (?=Statement|Example)"
parts = re.split(ITEM_ID, text)  # [intro, id, text, id, text, ...]
chunks = pd.DataFrame({"id": parts[1::2], "text": [f"{i} {t.strip()}" for i, t in zip(parts[1::2], parts[2::2])]})
assert chunks.id.is_unique  # every item id is its own chunk, so gold ranks are unambiguous
print(len(chunks), "chunks | words: median", chunks.text.str.split().str.len().median(), "max", chunks.text.str.split().str.len().max())

583 chunks | words: median 85.0 max 455


## 3. Models and index (chunk embeddings are computed once)

In [3]:
bi = SentenceTransformer(BI, model_kwargs={"torch_dtype": torch.bfloat16})
ce = CrossEncoder(CE, model_kwargs={"torch_dtype": torch.bfloat16})
llm = OpenAI(base_url=LLM_URL, api_key="EMPTY")
doc_emb = bi.encode_document(chunks.text.tolist(), batch_size=4, normalize_embeddings=True)
print(doc_emb.shape)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

(583, 2560)


## 4. Pipelines: P1 = cosine top-10 → top-3; P2 = cross-encoder re-sorts the same 10 → top-3; same prompt

In [4]:
PROMPT = """Answer the question using ONLY the context. Cite the item ids you used, e.g. (12-26).
If the answer is not in the context, say "Not in context." Max 4 sentences.

Context:
{context}

Question: {question}"""

def retrieve(q, k=10):  # Pipeline 1: bi-encoder cosine top-k
    return np.argsort(-(doc_emb @ bi.encode_query(q, normalize_embeddings=True)))[:k]

def rerank(q, idx):  # Pipeline 2: cross-encoder scores each (query, chunk) pair
    return idx[np.argsort(-ce.predict([(q, chunks.text[i]) for i in idx]))]

def answer(q, idx, think):  # think=True lets Qwen reason first (vLLM returns the reasoning separately)
    r = llm.chat.completions.create(
        model="qwen", temperature=0, max_tokens=4096 if think else 400,
        messages=[{"role": "user", "content": PROMPT.format(context="\n\n".join(chunks.text[idx]), question=q)}],
        extra_body={"chat_template_kwargs": {"enable_thinking": think}})
    return (r.choices[0].message.content or "").strip()

## 5. Test questions (20): gold item id + gold answer, 5 per failure type

In [5]:
QUERIES = [  # (category, question, gold item id, gold answer)
    ("keyword_trap", "Can a player wear a scarf-type headband during a game?", "4-4",
     "No. Scarf-style headbands are not permitted."),
    ("keyword_trap", "Is it a violation if a player grabs the ring and makes the ball bounce out?", "31-17",
     "Yes. Causing the basket to vibrate or grasping it so the ball is kept out (or put in) is an interference violation."),
    ("keyword_trap", "What is the penalty when a player pretends to be fouled?", "36-12",
     "First a signal and a warning to the team; the next fake by that team is a technical foul."),
    ("keyword_trap", "How many challenges can a head coach request in one game?", "F-4.15",
     "Only one head coach's challenge (HCC) per game."),
    ("keyword_trap", "If a defender commits a foul after the shooter has started the shooting motion, does the basket count?", "10-1",
     "Yes. The ball stays live and the goal counts if the shooter finishes with a continuous motion."),
    ("exception", "Who gets the ball at a jump ball situation if the shot clock has run out and the arrow favours the team in control?", "12-26",
     "The opponents. It is a shot clock violation, so the alternating possession procedure is not used."),
    ("exception", "If a player is fouled while shooting and then travels before scoring, does the goal count?", "15-10",
     "No. The goal does not count and 2 or 3 free throws are awarded."),
    ("exception", "In the last two minutes of the game, can the coach choose where the throw-in is taken after a time-out?", "17-12",
     "Yes, with 2:00 or less in the 4th quarter/overtime and a backcourt throw-in, the head coach picks frontcourt throw-in line or backcourt."),
    ("exception", "How much shot clock time does a team get on an alternating possession throw-in if it already had the ball?", "29/50-12",
     "Only the time that was left on the shot clock when the jump ball situation happened."),
    ("exception", "What happens when the ball gets stuck between the ring and the backboard?", "12-16",
     "Jump ball situation: alternating possession throw-in (14 s for the offense, 24 s for the defense), except between free throws."),
    ("negation", "When can a time-out not be granted?", "18/19-1",
     "Before the playing time of a quarter or overtime has started, or after it has ended."),
    ("negation", "Can a substitute be taken out again right after entering the game?", "18/19-14",
     "No. Only after the end of the next game clock running period."),
    ("negation", "Can the thrower-in bounce the ball on the court and then be the first to touch it again?", "17-31",
     "No. Another player must touch the ball first."),
    ("negation", "Can a player step out of the court to avoid a 3-second violation and then come back into the restricted area?", "26-1",
     "No. Leaving at the endline to avoid 3 seconds and re-entering the restricted area is a violation."),
    ("negation", "Are defenders allowed to distract a shooter by shouting or waving hands near the eyes?", "36-3",
     "No. It is a technical foul if the shooter is disadvantaged, otherwise a warning."),
    ("buried", "How many seconds are on the shot clock when the offense gets the rebound after the ball hits the ring?", "29/50-35",
     "14 seconds, if the same team that had the ball before it touched the ring gains control."),
    ("buried", "Does the 8-second count continue after a jump ball situation if the same team gets the throw-in in its backcourt?", "28-1",
     "Yes. The 8-second period continues."),
    ("buried", "If a player who already has 5 fouls commits a technical foul, who is it charged to?", "37-8",
     "The head coach, recorded as a 'B' technical foul."),
    ("buried", "Can a time-out request be cancelled while an instant replay review is going on?", "F-2.5",
     "Only after the review ends and the referee communicates the final decision."),
    ("buried", "How many technical or unsportsmanlike fouls get a player automatically disqualified?", "36-19",
     "2 technical fouls, 2 unsportsmanlike fouls, or 1 of each."),
]

## 6. Run both pipelines (LLM in non-thinking and thinking mode), timing every stage

In [6]:
def timed(f, *args):
    t = time.perf_counter(); out = f(*args)
    return out, time.perf_counter() - t

rank = lambda idx, gold: next((r for r, i in enumerate(idx, 1) if chunks.id[i] == gold), None)  # 1-based, None = not in top-10
rerank("warm up", retrieve("warm up")); answer("warm up", [0], False)  # warm-up so timings are fair

rows = []
for cat, q, gold, gold_answer in QUERIES:
    p1, t_retrieve = timed(retrieve, q)
    p2, t_rerank = timed(rerank, q, p1)
    row = dict(category=cat, query=q, gold=gold, gold_answer=gold_answer, p1_rank=rank(p1, gold), p2_rank=rank(p2, gold),
               p1_top3=list(chunks.id[p1[:3]]), p2_top3=list(chunks.id[p2[:3]]), t_retrieve=t_retrieve, t_rerank=t_rerank)
    for mode, think in [("nothink", False), ("think", True)]:
        for p, idx in [("1", p1), ("2", p2)]:
            row[f"answer{p}_{mode}"], row[f"t_llm{p}_{mode}"] = timed(answer, q, idx[:3], think)
    rows.append(row)
res = pd.DataFrame(rows)

## 7. Rank shift: RESCUED = gold chunk enters the top-3 only after re-ranking

In [7]:
in3 = lambda r: pd.notna(r) and r <= 3
res["status"] = [("missed by retriever" if pd.isna(a) else {(False, True): "RESCUED", (True, False): "REGRESSED"}.get((in3(a), in3(b)), "same"))
                 for a, b in zip(res.p1_rank, res.p2_rank)]
for p in "12":
    for m in ["nothink", "think"]:
        res[f"cites_gold{p}_{m}"] = [g in a for g, a in zip(res.gold, res[f"answer{p}_{m}"])]  # proxy: answer cites the gold item
res.to_csv("results/results.csv", index=False)

mrr = lambda s: s.apply(lambda r: 0 if pd.isna(r) else 1 / r).mean()
print(res.status.value_counts().to_dict())
for p in "12":
    print(f"Pipeline {p}: gold in top-3 {res[f'p{p}_rank'].apply(in3).sum()}/20 | MRR@10 {mrr(res[f'p{p}_rank']):.3f} | "
          f"answers citing gold: non-thinking {res[f'cites_gold{p}_nothink'].mean():.0%}, thinking {res[f'cites_gold{p}_think'].mean():.0%}")
res[["category", "query", "gold", "p1_rank", "p2_rank", "status", "p1_top3", "p2_top3"]]

{'same': 14, 'RESCUED': 5, 'REGRESSED': 1}
Pipeline 1: gold in top-3 15/20 | MRR@10 0.730 | answers citing gold: non-thinking 70%, thinking 70%
Pipeline 2: gold in top-3 19/20 | MRR@10 0.824 | answers citing gold: non-thinking 80%, thinking 95%


,category,query,gold,p1_rank,p2_rank,status,p1_top3,p2_top3
0,keyword_trap,Can a player wear a scarf-type headband during...,4-4,2,2,same,"[4-5, 4-4, 4-2]","[4-5, 4-4, 4-8]"
1,keyword_trap,Is it a violation if a player grabs the ring a...,31-17,3,1,same,"[31-20, 31-23, 31-17]","[31-17, 31-22, 31-7]"
2,keyword_trap,What is the penalty when a player pretends to ...,36-12,1,1,same,"[36-12, 36-16, 36-15]","[36-12, 36-28, 36-16]"
3,keyword_trap,How many challenges can a head coach request i...,F-4.15,5,1,RESCUED,"[F-4.1, F-4.22, F-4.17]","[F-4.15, F-4.1, F-4.3]"
4,keyword_trap,If a defender commits a foul after the shooter...,10-1,10,1,RESCUED,"[15-4, 15-7, 8-2]","[10-1, 15-10, 15-7]"
5,exception,Who gets the ball at a jump ball situation if ...,12-26,1,1,same,"[12-26, 12-18, 28-1]","[12-26, 29/50-13, 12-20]"
6,exception,If a player is fouled while shooting and then ...,15-10,1,1,same,"[15-10, 15-11, 17-51]","[15-10, 17-51, 10-1]"
7,exception,"In the last two minutes of the game, can the c...",17-12,1,1,same,"[17-12, 17-21, 17-13]","[17-12, 17-21, 17-15]"
8,exception,How much shot clock time does a team get on an...,29/50-12,1,7,REGRESSED,"[29/50-12, 29/50-13, 12-17]","[12-16, 29/50-13, 29/50-42]"
9,exception,What happens when the ball gets stuck between ...,12-16,8,1,RESCUED,"[31-20, 17-30, 31-3]","[12-16, 12-20, 17-28]"


## 8. Answer 1 vs Answer 2 where the top-3 changed

In [8]:
for _, r in res[res.status.isin(["RESCUED", "REGRESSED"])].iterrows():
    print(f"\n[{r.status}] {r.query}\nGOLD ({r.gold}): {r.gold_answer}  | rank #{r.p1_rank:.0f} -> #{r.p2_rank:.0f}")
    for m in ["nothink", "think"]:
        print(f"  {m:8} A1 {r.p1_top3}: {r[f'answer1_{m}'] or '(empty)'}")
        print(f"  {m:8} A2 {r.p2_top3}: {r[f'answer2_{m}'] or '(empty)'}")


[RESCUED] How many challenges can a head coach request in one game?
GOLD (F-4.15): Only one head coach's challenge (HCC) per game.  | rank #5 -> #1
  nothink  A1 ['F-4.1', 'F-4.22', 'F-4.17']: Not in context.
  nothink  A2 ['F-4.15', 'F-4.1', 'F-4.3']: In all games where the Instant Replay System (IRS) is used, the head coach may be granted only one HCC (12-26).
  think    A1 ['F-4.1', 'F-4.22', 'F-4.17']: Not in context.
  think    A2 ['F-4.15', 'F-4.1', 'F-4.3']: A head coach may be granted only one HCC in a game where the Instant Replay System is used (F-4.15).

[RESCUED] If a defender commits a foul after the shooter has started the shooting motion, does the basket count?
GOLD (10-1): Yes. The ball stays live and the goal counts if the shooter finishes with a continuous motion.  | rank #10 -> #1
  nothink  A1 ['15-4', '15-7', '8-2']: Yes, if the defender commits a foul after the shooter has started the shooting motion, the basket counts if the shot is successful. In this scenario,

## 9. Latency (ms, mean per question)

In [9]:
ms = lambda c: round(res[c].mean() * 1000, 1)
lat = pd.DataFrame({"Pipeline 1": [ms("t_retrieve"), 0, ms("t_llm1_nothink"), ms("t_llm1_think")],
                    "Pipeline 2": [ms("t_retrieve"), ms("t_rerank"), ms("t_llm2_nothink"), ms("t_llm2_think")]},
                   index=["bi-encoder retrieve", "cross-encoder rerank", "LLM non-thinking", "LLM thinking"])
lat.to_csv("results/latency.csv")
lat

,Pipeline 1,Pipeline 2
bi-encoder retrieve,78.2,78.2
cross-encoder rerank,0.0,43.6
LLM non-thinking,633.4,742.7
LLM thinking,5772.7,5272.6


## 10. Scaling: cross-encoder over every chunk vs re-ranking only 10

In [10]:
_, t_all = timed(lambda pairs: ce.predict(pairs, batch_size=8), [(QUERIES[0][1], t) for t in chunks.text])
per_pair = t_all / len(chunks)
print(f"cross-encoder on all {len(chunks)} chunks: {t_all * 1000:.0f} ms ({per_pair * 1000:.2f} ms/pair) vs {ms('t_rerank')} ms for 10")
print(f"1M chunks: ~{per_pair * 1e6 / 60:.0f} min per question")

cross-encoder on all 583 chunks: 2063 ms (3.54 ms/pair) vs 43.6 ms for 10
1M chunks: ~59 min per question
